# Comparación de Trials que Cambiaron de Validez

Este notebook compara lado a lado los 34 trials que cambiaron su estado de validez entre el análisis raw e interpolado.

In [7]:
import sys
sys.path.insert(0, '/home/gianluca/Research/datapruebas_analysis')

from pathlib import Path
import pandas as pd

# Fechas de los análisis
INTERPOLATED_DATE = "2026-01-23_09-24-48"
RAW_DATE = "2026-01-23_09-24-13"

# Rutas
BASE_PATH = Path('/home/gianluca/Research/datapruebas_analysis/data/hand_analysis')
interp_path = BASE_PATH / INTERPOLATED_DATE / 'analysis.csv'
raw_path = BASE_PATH / RAW_DATE / 'analysis.csv'

In [8]:
# Cargar análisis completos (incluyendo trials inválidos)
df_interp = pd.read_csv(interp_path)
df_raw = pd.read_csv(raw_path)

print(f"Interpolado: {len(df_interp):,} trials ({df_interp['is_valid'].sum():,} válidos)")
print(f"Raw: {len(df_raw):,} trials ({df_raw['is_valid'].sum():,} válidos)")

Interpolado: 7,508 trials (5,668 válidos)
Raw: 7,508 trials (5,654 válidos)


In [9]:
# Merge para comparar lado a lado (incluir error_message para ver conteo real)
cols_to_compare = ['subject_id', 'trial_id', 'is_valid', 'invalid_cause', 'error_message', 'correct_targets_touches']

df_merged = pd.merge(
    df_raw[cols_to_compare],
    df_interp[cols_to_compare],
    on=['subject_id', 'trial_id'],
    suffixes=('_raw', '_interp')
)

# Filtrar trials donde cambió la validez
df_changed = df_merged[df_merged['is_valid_raw'] != df_merged['is_valid_interp']].copy()

print(f"Trials que cambiaron de validez: {len(df_changed)}")

Trials que cambiaron de validez: 34


In [10]:
# Extraer conteo de targets del error_message
import re

def extract_target_count(msg):
    if pd.isna(msg):
        return None
    match = re.search(r'has (\d+) correct target touches', str(msg))
    return int(match.group(1)) if match else None

df_changed['targets_raw'] = df_changed.apply(
    lambda r: int(r['correct_targets_touches_raw']) if r['is_valid_raw'] else extract_target_count(r['error_message_raw']),
    axis=1
)
df_changed['targets_interp'] = df_changed.apply(
    lambda r: int(r['correct_targets_touches_interp']) if r['is_valid_interp'] else extract_target_count(r['error_message_interp']),
    axis=1
)

# Ordenar: primero los que se volvieron válidos, luego por targets_raw
df_changed = df_changed.sort_values(['is_valid_interp', 'targets_raw'], ascending=[False, True])

# Mostrar tabla con columnas relevantes
df_changed[['subject_id', 'trial_id', 'is_valid_raw', 'targets_raw', 'is_valid_interp', 'targets_interp']]

,subject_id,trial_id,is_valid_raw,targets_raw,is_valid_interp,targets_interp
280,b1368f7d-47b5-45dd-83f4-5167eca84158,NEUROPRUEBAS_2,False,0,True,10
876,1aadc486-9800-46c6-93f6-13aa11168a00,NEUROPRUEBAS_12,False,0,True,10
929,b4dc8afd-c6ae-4335-9694-217a9e6f2e05,NEUROPRUEBAS_5,False,0,True,10
936,b4dc8afd-c6ae-4335-9694-217a9e6f2e05,NEUROPRUEBAS_12,False,0,True,10
1349,26815196-070f-4f94-a765-5bbecabea52c,NEUROPRUEBAS_10,False,0,True,10
1821,b65df3e7-91c6-40b7-a732-49622bfa8684,NEUROPRUEBAS_10,False,0,True,10
3697,4657de88-e8fe-4eca-b171-366b1822d2b3,NEUROPRUEBAS_5,False,0,True,10
4220,43fab341-f3f2-4c51-a9e6-81cd008f96d9,NEUROPRUEBAS_9,False,0,True,10
5325,2fbfdfa5-3297-42e0-bca4-b21203ab2596,NEUROPRUEBAS_20,False,0,True,10
5375,a074d1e7-c9ac-440a-8397-7f68f6c94ffe,NEUROPRUEBAS_10,False,0,True,10


In [11]:
# Resumen de cambios en conteo de targets
raw_to_valid = df_changed[~df_changed['is_valid_raw'] & df_changed['is_valid_interp']]
valid_to_invalid = df_changed[df_changed['is_valid_raw'] & ~df_changed['is_valid_interp']]

print("=" * 60)
print("INVÁLIDO (raw) → VÁLIDO (interp): 24 trials")
print("=" * 60)
print(f"{'targets_raw':<15} {'targets_interp':<15} {'cantidad':<10}")
print("-" * 40)
for targets_raw, group in raw_to_valid.groupby('targets_raw'):
    targets_interp = int(group['targets_interp'].iloc[0])
    print(f"{targets_raw:<15} {targets_interp:<15} {len(group):<10}")

print()
print("=" * 60)
print("VÁLIDO (raw) → INVÁLIDO (interp): 10 trials")
print("=" * 60)
print(f"{'targets_raw':<15} {'targets_interp':<15} {'cantidad':<10}")
print("-" * 40)
for targets_interp, group in valid_to_invalid.groupby('targets_interp'):
    targets_raw = int(group['targets_raw'].iloc[0])
    print(f"{targets_raw:<15} {targets_interp:<15} {len(group):<10}")

print()
print("=" * 60)
print(f"EFECTO NETO: {len(raw_to_valid) - len(valid_to_invalid):+d} trials")
print("=" * 60)

INVÁLIDO (raw) → VÁLIDO (interp): 24 trials
targets_raw     targets_interp  cantidad  
----------------------------------------
0               10              16        
1               10              1         
2               10              1         
3               10              1         
4               10              1         
7               10              3         
9               10              1         

VÁLIDO (raw) → INVÁLIDO (interp): 10 trials
targets_raw     targets_interp  cantidad  
----------------------------------------
10              0               1         
10              1               1         
10              2               1         
10              3               2         
10              4               1         
10              5               3         
10              8               1         

EFECTO NETO: +14 trials


## Visualización de trials que cambiaron de Válido → Inválido

In [12]:
import matplotlib.pyplot as plt

# Trials que cambiaron de válido (raw) a inválido (interp)
valid_to_invalid = df_changed[df_changed['is_valid_raw'] & ~df_changed['is_valid_interp']]

print(f"Plotear {len(valid_to_invalid)} trials que pasaron de válido → inválido")
print(valid_to_invalid[['subject_id', 'trial_id', 'targets_raw', 'targets_interp']].to_string(index=False))

# Función para encontrar subject y trial en el experimento
def find_subject_and_trial(experiments, subject_id, trial_id):
    """Buscar subject y trial en los experimentos."""
    for exp in experiments:
        if subject_id in exp.subjects:
            subject = exp.subjects[subject_id]
            for trial in subject.testing_trials:
                if trial.id == trial_id:
                    return subject, trial
    return None, None

Plotear 10 trials que pasaron de válido → inválido
                                  subject_id        trial_id  targets_raw  targets_interp
leandrogori2000@gmail.com-tmt-plugin_112.csv  NEUROPRUEBAS_2           10               8
        32d611d1-ad54-4835-99ed-460a443df14c  NEUROPRUEBAS_3           10               0
        48482d6a-a3ca-4c42-bdab-9462f26e6ff8 NEUROPRUEBAS_17           10               3
        4657de88-e8fe-4eca-b171-366b1822d2b3 NEUROPRUEBAS_20           10               2
        b17c4fc1-95ed-47b9-8dd5-72ee8705a62c NEUROPRUEBAS_21           10               5
        3cf3fc83-1ee9-4389-9978-fa5845035019 NEUROPRUEBAS_21           10               5
        2697d032-ba2c-45e4-8115-588455f5b491 NEUROPRUEBAS_10           10               5
        64bbfefd-5852-4921-8c79-c4f708b9b60a   DATAPRUEBAS_4           10               3
        961c65a9-5caa-43c3-8cc4-c4c5484cd1cc   DATAPRUEBAS_8           10               1
        139efc97-c335-401c-a98f-dd3825ab2d87  DAT

## Comparación lado a lado: Raw vs Interpolado

In [13]:
from src import config
from importlib import reload

# Cargar SIN interpolación
config.INTERPOLATE_TRAJECTORY = False
import src.visualization.plot_trials_cli as plot_cli
reload(plot_cli)
exp_raw_dp = plot_cli.load_experiment("datapruebas")
exp_raw_np = plot_cli.load_experiment("neuropruebas")

print(f"Raw - Datapruebas: {len(exp_raw_dp.subjects)} sujetos")
print(f"Raw - Neuropruebas: {len(exp_raw_np.subjects)} sujetos")

Mapping 127 subjects...


ERROR:root:Error processing experiment for subject f7031def-b705-41e7-aa54-da7e0c4ee601
Traceback (most recent call last):
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 64, in map_to_experiment
    subjects[experiment.subject_id] = self.map_to_subject(experiment.records[0], start_date)
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 94, in map_to_subject
    raise ValueError("Subject must have both training and testing stimuli")
ValueError: Subject must have both training and testing stimuli
ERROR:root:Error processing experiment for subject c890dfc5-aea9-4f93-833e-988009c44d71
Traceback (most recent call last):
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 64, in map_to_experiment
    subjects[experiment.subject_id] = self.map_to_subject(experiment.records[0], start_date)
  File "/home/gianluca/Research/dataprueb

Mapped 58 valid subjects.


ERROR:root:Error processing experiment for subject b7701e5e-85c1-43d1-8617-ab4944c95d48
Traceback (most recent call last):
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/neuropruebas/neuropruebas_mapper.py", line 145, in map_to_experiment
    subjects[subject_id] = self.map_to_subject(subject_id, subject_data, session_data)
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/neuropruebas/neuropruebas_mapper.py", line 234, in map_to_subject
    testing_trials, training_trials = self.map_to_testing_training_trials(subject_data, testing_stimuli,
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/neuropruebas/neuropruebas_mapper.py", line 250, in map_to_testing_training_trials
    self._extract_position_and_time_data(subject_data, training_stimuli, testing_stimuli))
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/neuropruebas/neuropruebas_mapper.py", line 279, in _extract_position_and_time_data
    raise ValueError("Subject csv does

Raw - Datapruebas: 58 sujetos
Raw - Neuropruebas: 355 sujetos


In [14]:
# Cargar CON interpolación
config.INTERPOLATE_TRAJECTORY = True
reload(plot_cli)
exp_interp_dp = plot_cli.load_experiment("datapruebas")
exp_interp_np = plot_cli.load_experiment("neuropruebas")

print(f"Interp - Datapruebas: {len(exp_interp_dp.subjects)} sujetos")
print(f"Interp - Neuropruebas: {len(exp_interp_np.subjects)} sujetos")

ERROR:root:Error processing experiment for subject f7031def-b705-41e7-aa54-da7e0c4ee601
Traceback (most recent call last):
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 64, in map_to_experiment
    subjects[experiment.subject_id] = self.map_to_subject(experiment.records[0], start_date)
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 94, in map_to_subject
    raise ValueError("Subject must have both training and testing stimuli")
ValueError: Subject must have both training and testing stimuli
ERROR:root:Error processing experiment for subject c890dfc5-aea9-4f93-833e-988009c44d71
Traceback (most recent call last):
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 64, in map_to_experiment
    subjects[experiment.subject_id] = self.map_to_subject(experiment.records[0], start_date)
  File "/home/gianluca/Research/dataprueb

Mapping 127 subjects...


ERROR:root:Error processing experiment for subject 139efc97-c335-401c-a98f-dd3825ab2d87
Traceback (most recent call last):
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 64, in map_to_experiment
    subjects[experiment.subject_id] = self.map_to_subject(experiment.records[0], start_date)
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 94, in map_to_subject
    raise ValueError("Subject must have both training and testing stimuli")
ValueError: Subject must have both training and testing stimuli
ERROR:root:Error processing experiment for subject d93d5b6d-7206-429f-80d0-3bb78b461c5b
Traceback (most recent call last):
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/datapruebas/datapruebas_mapper.py", line 64, in map_to_experiment
    subjects[experiment.subject_id] = self.map_to_subject(experiment.records[0], start_date)
  File "/home/gianluca/Research/dataprueb

Mapped 58 valid subjects.


ERROR:root:Error processing experiment for subject b7701e5e-85c1-43d1-8617-ab4944c95d48
Traceback (most recent call last):
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/neuropruebas/neuropruebas_mapper.py", line 145, in map_to_experiment
    subjects[subject_id] = self.map_to_subject(subject_id, subject_data, session_data)
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/neuropruebas/neuropruebas_mapper.py", line 234, in map_to_subject
    testing_trials, training_trials = self.map_to_testing_training_trials(subject_data, testing_stimuli,
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/neuropruebas/neuropruebas_mapper.py", line 250, in map_to_testing_training_trials
    self._extract_position_and_time_data(subject_data, training_stimuli, testing_stimuli))
  File "/home/gianluca/Research/datapruebas_analysis/src/mapper/neuropruebas/neuropruebas_mapper.py", line 279, in _extract_position_and_time_data
    raise ValueError("Subject csv does

Interp - Datapruebas: 58 sujetos
Interp - Neuropruebas: 355 sujetos


In [ ]:
from src.visualization.trial_plotting_labels import plot_trial_simple

# Plot comparativo lado a lado
fig, axes = plt.subplots(10, 2, figsize=(12, 40))

for idx, (_, row) in enumerate(valid_to_invalid.iterrows()):
    subject_id = row['subject_id']
    trial_id = row['trial_id']
    targets_raw = row['targets_raw']
    targets_interp = row['targets_interp']
    
    # Buscar en raw
    subj_raw, trial_raw = find_subject_and_trial([exp_raw_dp, exp_raw_np], subject_id, trial_id)
    # Buscar en interpolado
    subj_interp, trial_interp = find_subject_and_trial([exp_interp_dp, exp_interp_np], subject_id, trial_id)
    
    # Plot raw (izquierda)
    plot_trial_simple(axes[idx, 0], trial_raw, subj_raw.target_radius, f"RAW - {trial_id}\ntargets: {targets_raw}")
    
    # Plot interpolado (derecha)
    plot_trial_simple(axes[idx, 1], trial_interp, subj_interp.target_radius, f"INTERP - {trial_id}\ntargets: {targets_interp}")

plt.suptitle("Comparación: Raw (izq) vs Interpolado (der)", fontsize=14, y=1.001)
plt.tight_layout()
plt.show()